# Intelligent Municipal Water Contamination & Infrastructure Alert System
### SDG 6 — Clean Water and Sanitation
Run each cell in order (Shift+Enter). Built entirely with free tools: Colab, scikit-learn, pandas, smtplib.


## Step 1: Import Libraries

In [ ]:
# ---- Core data handling ----
import pandas as pd
import numpy as np

# ---- Visualization ----
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Machine learning ----
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import joblib

# ---- Utility ----
import uuid
import json
import time
from datetime import datetime, timedelta

# ---- Email alerts (built into Python) ----
import smtplib
from email.mime.text import MIMEText

# ---- Colab upload/download helper ----
from google.colab import files

print("All libraries imported successfully.")

## Step 2: Upload Your Kaggle Dataset\nRun this cell, then click **Choose Files** and select `water_potability.csv`.

In [ ]:
uploaded = files.upload()  # select water_potability.csv when prompted
df_raw = pd.read_csv("water_potability.csv")
print(df_raw.shape)
df_raw.head()

## Step 3: Inspect & Handle Missing Values

In [ ]:
print(df_raw.isna().sum())

df = df_raw.copy()
for col in ["ph", "Sulfate", "Trihalomethanes"]:
    df[col] = df.groupby("Potability")[col].transform(lambda x: x.fillna(x.median()))

print("\nRemaining missing values:")
print(df.isna().sum())

## Step 4: Transform Into a Simulated IoT Sensor Stream
This dataset has no timestamp, node, GPS, dissolved oxygen, or flow rate — so we synthesize those
to match a real pipeline-monitoring schema, while keeping `ph`, `Turbidity`, and `Potability` as
the real signal driving contamination anomalies.

In [ ]:
np.random.seed(42)
N_NODES = 10
n_rows = len(df)

# Assign each row to a fake pipeline node
df["node_id"] = np.random.choice([f"NODE_{i+1:02d}" for i in range(N_NODES)], size=n_rows)

# Fake GPS coordinates per node, clustered around a city (example: Chicago)
city_lat, city_lon = 41.8781, -87.6298
node_gps = {
    f"NODE_{i+1:02d}": (
        round(city_lat + np.random.uniform(-0.05, 0.05), 6),
        round(city_lon + np.random.uniform(-0.05, 0.05), 6),
    )
    for i in range(N_NODES)
}
df["gps_lat"] = df["node_id"].map(lambda n: node_gps[n][0])
df["gps_lon"] = df["node_id"].map(lambda n: node_gps[n][1])

# Synthetic timestamp: space rows 15 minutes apart within each node
df = df.sort_values("node_id").reset_index(drop=True)
start_time = datetime(2026, 1, 1)
df["reading_order"] = df.groupby("node_id").cumcount()
df["timestamp"] = df.apply(
    lambda r: start_time + timedelta(minutes=15 * r["reading_order"]), axis=1
)

# Synthetic dissolved oxygen — lower when Potability == 0 (contaminated)
df["dissolved_oxygen_mg_L"] = np.where(
    df["Potability"] == 0,
    np.random.uniform(3.0, 6.0, n_rows),
    np.random.uniform(6.0, 9.0, n_rows),
)

# Synthetic flow rate per node, with ~2% sudden drops simulating pipe bursts
node_baseline_flow = {n: np.random.uniform(20, 40) for n in node_gps}
df["flow_rate_Lps"] = df["node_id"].map(node_baseline_flow) + np.random.normal(0, 1.5, n_rows)
leak_mask = np.random.rand(n_rows) < 0.02
df.loc[leak_mask, "flow_rate_Lps"] = df.loc[leak_mask, "flow_rate_Lps"] * np.random.uniform(0.1, 0.3)

# Ground-truth anomaly label
df["anomaly_type"] = "none"
df.loc[df["Potability"] == 0, "anomaly_type"] = "contamination"
df.loc[leak_mask, "anomaly_type"] = "leak"

# Final schema
water_sensor_data = df[[
    "timestamp", "node_id", "gps_lat", "gps_lon",
    "ph", "Turbidity", "dissolved_oxygen_mg_L", "flow_rate_Lps", "anomaly_type"
]].rename(columns={"ph": "pH", "Turbidity": "turbidity_NTU"})

water_sensor_data.to_csv("water_sensor_data.csv", index=False)
print(water_sensor_data["anomaly_type"].value_counts())
water_sensor_data.head()

## Step 5: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sensors = ["pH", "turbidity_NTU", "dissolved_oxygen_mg_L", "flow_rate_Lps"]
colors = {"none": "steelblue", "contamination": "red", "leak": "orange"}

for ax, sensor in zip(axes.flat, sensors):
    for label, group in water_sensor_data.groupby("anomaly_type"):
        ax.scatter(group.index, group[sensor], s=6, label=label, color=colors[label], alpha=0.6)
    ax.set_title(sensor)
    ax.legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(water_sensor_data[sensors].corr(), annot=True, cmap="coolwarm")
plt.title("Sensor Correlation Heatmap")
plt.show()

## Step 6: Feature Engineering & Train/Test Split

In [ ]:
features = ["pH", "turbidity_NTU", "dissolved_oxygen_mg_L", "flow_rate_Lps"]
X = water_sensor_data[features]
y = water_sensor_data["anomaly_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Step 7: Train the Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=rf_model.classes_, yticklabels=rf_model.classes_, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest")
plt.show()

joblib.dump(rf_model, "water_anomaly_model.pkl")
print("Model saved as water_anomaly_model.pkl")

## Step 8: Real-Time Scoring Function\nThis is what the automation pipeline calls on every new reading.

In [ ]:
model = joblib.load("water_anomaly_model.pkl")

def check_reading(sensor_reading: dict) -> dict:
    """
    sensor_reading example:
    {"node_id": "NODE_01", "gps_lat": 41.88, "gps_lon": -87.63,
     "pH": 4.2, "turbidity_NTU": 55, "dissolved_oxygen_mg_L": 3.1,
     "flow_rate_Lps": 25, "timestamp": "2026-01-01T00:00:00"}
    """
    X_new = pd.DataFrame([{
        "pH": sensor_reading["pH"],
        "turbidity_NTU": sensor_reading["turbidity_NTU"],
        "dissolved_oxygen_mg_L": sensor_reading["dissolved_oxygen_mg_L"],
        "flow_rate_Lps": sensor_reading["flow_rate_Lps"],
    }])

    pred = model.predict(X_new)[0]
    proba = model.predict_proba(X_new)[0]
    confidence = float(max(proba))

    return {
        "is_anomaly": pred != "none",
        "anomaly_type": pred,
        "confidence_score": round(confidence, 3),
        "node_id": sensor_reading["node_id"],
        "gps_coordinates": (sensor_reading["gps_lat"], sensor_reading["gps_lon"]),
        "reading": sensor_reading,
    }

# Quick test
test_reading = water_sensor_data[water_sensor_data["anomaly_type"] != "none"].iloc[0].to_dict()
print(check_reading(test_reading))

## Step 9: Automation — Emergency Shutdown Signal

In [ ]:
def trigger_shutdown(node_id, gps_coords, reading):
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "node_id": node_id,
        "gps_coords": gps_coords,
        "reason": reading.get("anomaly_type", "unknown"),
        "status": "SIGNAL_SENT",
    }
    log_df = pd.DataFrame([log_entry])
    log_df.to_csv("shutdown_log.csv", mode="a",
                   header=not pd.io.common.file_exists("shutdown_log.csv"), index=False)
    print(f"[SHUTDOWN] Valve at {node_id} closing... signal sent. Coordinates: {gps_coords}")
    return log_entry

## Step 10: Automation — Auto-Generate Maintenance Ticket

In [ ]:
def generate_ticket(node_id, gps_coords, reading, anomaly_type):
    ticket = {
        "ticket_id": str(uuid.uuid4()),
        "priority": "CRITICAL" if anomaly_type == "contamination" else "HIGH",
        "node_id": node_id,
        "gps_coordinates": gps_coords,
        "reading": reading,
        "anomaly_type": anomaly_type,
        "timestamp": datetime.now().isoformat(),
        "status": "OPEN",
    }
    filename = f"ticket_{ticket['ticket_id'][:8]}.json"
    with open(filename, "w") as f:
        json.dump(ticket, f, indent=2, default=str)
    print(f"[TICKET] {filename} created — priority {ticket['priority']}")
    return ticket

## Step 11: Automation — Citizen/Authority Alert Blast
Email uses Gmail's free SMTP. To use it: enable 2-Step Verification on your Gmail account,
then create an **App Password** at myaccount.google.com/apppasswords and paste it below.
Leave `SEND_REAL_EMAIL = False` to just simulate/log alerts without sending anything.

In [ ]:
SEND_REAL_EMAIL = False  # set True once you've added real credentials below
GMAIL_ADDRESS = "your_email@gmail.com"
GMAIL_APP_PASSWORD = "your_16_char_app_password"

def send_alerts(node_id, gps_coords, anomaly_type, affected_contacts: list):
    message_body = (
        f"WATER ALERT: A {anomaly_type} event was detected near pipeline node {node_id} "
        f"(coordinates {gps_coords}). Please avoid using tap water in this area until "
        f"authorities confirm it is safe. Local water authority has been notified."
    )

    if not SEND_REAL_EMAIL:
        print(f"[ALERT - SIMULATED] To: {affected_contacts}\n{message_body}")
        return {"status": "simulated", "recipients": affected_contacts}

    msg = MIMEText(message_body)
    msg["Subject"] = f"Water Safety Alert — {anomaly_type.upper()} near Node {node_id}"
    msg["From"] = GMAIL_ADDRESS
    msg["To"] = ", ".join(affected_contacts)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_ADDRESS, affected_contacts, msg.as_string())

    print(f"[ALERT - SENT] Email sent to {affected_contacts}")
    return {"status": "sent", "recipients": affected_contacts}

## Step 12: End-to-End Pipeline Simulation

In [ ]:
def simulate_pipeline(data: pd.DataFrame, n_readings=30, delay_seconds=0.3,
                       affected_contacts=["citizen_demo@example.com"]):
    pipeline_log = []
    sample = data.sample(n_readings, random_state=1).to_dict("records")

    for reading in sample:
        result = check_reading(reading)
        status_line = f"Node {result['node_id']}: "

        if result["is_anomaly"]:
            status_line += f"ANOMALY ({result['anomaly_type']}, conf={result['confidence_score']})"
            trigger_shutdown(result["node_id"], result["gps_coordinates"], result)
            generate_ticket(result["node_id"], result["gps_coordinates"], reading, result["anomaly_type"])
            send_alerts(result["node_id"], result["gps_coordinates"], result["anomaly_type"], affected_contacts)
        else:
            status_line += "normal"

        print(status_line)
        pipeline_log.append({**result, "reading": str(result["reading"])})
        time.sleep(delay_seconds)

    pd.DataFrame(pipeline_log).to_csv("pipeline_log.csv", index=False)
    print("\nPipeline run complete. Log saved to pipeline_log.csv")

simulate_pipeline(water_sensor_data, n_readings=30)

## Step 13: Download Your Generated Files (optional)

In [ ]:
for fname in ["water_sensor_data.csv", "water_anomaly_model.pkl", "pipeline_log.csv", "shutdown_log.csv"]:
    try:
        files.download(fname)
    except Exception as e:
        print(f"Could not download {fname}: {e}")

## Next Step
Once this runs end-to-end, the next add-on is a **Streamlit dashboard** (map view of nodes +
live charts + open tickets table) — that's a separate script since Streamlit doesn't run
inside Colab cells directly. Just ask when you're ready for it.